# Generic Toxic Plume Simulator Notebook (Single-Notebook Version)


In [2]:
! pip install numpy

zsh:1: command not found: pip


In [3]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from dataclasses import dataclass

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

print('SUCCEEDED: Imports loaded')


ModuleNotFoundError: No module named 'numpy'

In [ ]:
@dataclass
class PlumeConfig:
    x_min_km: float = -25.0
    x_max_km: float = 25.0
    y_min_km: float = -20.0
    y_max_km: float = 20.0
    resolution: int = 320
    source_x_km: float = 0.0
    source_y_km: float = 0.0
    wind_from_deg: float = 270.0
    wind_speed_mps: float = 5.0
    emission_rate: float = 1200.0
    release_height_m: float = 10.0
    stability_class: str = 'D'
    min_downwind_m: float = 10.0
    threshold_low: float = 0.5
    threshold_medium: float = 2.0
    threshold_high: float = 5.0

cfg = PlumeConfig()

FICTIONAL_NEIGHBORHOODS = {
    'Northgate': (2.0, 8.0),
    'Eastfield': (11.0, 1.0),
    'Westpark': (-9.5, -2.5),
    'Southbank': (1.0, -10.0),
    'Old Town': (5.0, 4.0),
    'Hillview': (-4.0, 9.0),
    'Riverside': (8.0, -6.5),
    'Market Row': (-1.0, 2.0),
}

print('SUCCEEDED: Configuration created')


In [ ]:
def make_grid(cfg: PlumeConfig):
    xs = np.linspace(cfg.x_min_km, cfg.x_max_km, cfg.resolution)
    ys = np.linspace(cfg.y_min_km, cfg.y_max_km, cfg.resolution)
    X_km, Y_km = np.meshgrid(xs, ys)
    return X_km, Y_km

def wind_to_math_angle_rad(wind_from_deg: float) -> float:
    wind_to_deg = (wind_from_deg + 180.0) % 360.0
    return np.deg2rad(wind_to_deg)

def rotate_to_wind_frame(X_km, Y_km, source_x_km, source_y_km, wind_from_deg):
    theta = wind_to_math_angle_rad(wind_from_deg)
    dx_m = (X_km - source_x_km) * 1000.0
    dy_m = (Y_km - source_y_km) * 1000.0
    x_prime = np.cos(theta) * dx_m + np.sin(theta) * dy_m
    y_prime = -np.sin(theta) * dx_m + np.cos(theta) * dy_m
    return x_prime, y_prime

def sigma_yz(x_m, stability_class='D'):
    x = np.maximum(x_m, 1.0)
    cls = stability_class.upper()
    if cls == 'A':
        sy = 0.22 * x * (1 + 0.0001 * x) ** (-0.5)
        sz = 0.20 * x
    elif cls == 'B':
        sy = 0.16 * x * (1 + 0.0001 * x) ** (-0.5)
        sz = 0.12 * x
    elif cls == 'C':
        sy = 0.11 * x * (1 + 0.0001 * x) ** (-0.5)
        sz = 0.08 * x * (1 + 0.0002 * x) ** (-0.5)
    elif cls == 'D':
        sy = 0.08 * x * (1 + 0.0001 * x) ** (-0.5)
        sz = 0.06 * x * (1 + 0.0015 * x) ** (-0.5)
    elif cls == 'E':
        sy = 0.06 * x * (1 + 0.0001 * x) ** (-0.5)
        sz = 0.03 * x * (1 + 0.0003 * x) ** (-1.0)
    elif cls == 'F':
        sy = 0.04 * x * (1 + 0.0001 * x) ** (-0.5)
        sz = 0.016 * x * (1 + 0.0003 * x) ** (-1.0)
    else:
        raise ValueError('stability_class must be one of A, B, C, D, E, F')
    sy = np.maximum(sy, 1.0)
    sz = np.maximum(sz, 1.0)
    return sy, sz

def gaussian_plume_ground_level(x_m, y_m, Q, U, H, stability_class='D', min_downwind_m=10.0):
    C = np.zeros_like(x_m, dtype=float)
    mask = x_m > min_downwind_m
    if not np.any(mask):
        return C
    x_pos = x_m[mask]
    y_pos = y_m[mask]
    sy, sz = sigma_yz(x_pos, stability_class)
    prefactor = Q / (np.pi * U * sy * sz)
    crosswind = np.exp(-(y_pos ** 2) / (2.0 * sy ** 2))
    vertical = np.exp(-(H ** 2) / (2.0 * sz ** 2))
    C[mask] = prefactor * crosswind * vertical
    return C

def compute_concentration(cfg: PlumeConfig):
    X_km, Y_km = make_grid(cfg)
    x_m, y_m = rotate_to_wind_frame(X_km, Y_km, cfg.source_x_km, cfg.source_y_km, cfg.wind_from_deg)
    C = gaussian_plume_ground_level(
        x_m=x_m,
        y_m=y_m,
        Q=cfg.emission_rate,
        U=max(cfg.wind_speed_mps, 0.1),
        H=cfg.release_height_m,
        stability_class=cfg.stability_class,
        min_downwind_m=cfg.min_downwind_m,
    )
    return X_km, Y_km, C

def concentration_band(C, cfg: PlumeConfig):
    if C >= cfg.threshold_high:
        return 'HIGH'
    elif C >= cfg.threshold_medium:
        return 'MEDIUM'
    elif C >= cfg.threshold_low:
        return 'LOW'
    return 'MINIMAL'

print('SUCCEEDED: Helper functions created')


In [ ]:
def plot_plume(cfg: PlumeConfig, neighborhoods=None, log_scale=True, title_suffix=''):
    X_km, Y_km, C = compute_concentration(cfg)
    fig, ax = plt.subplots(figsize=(11, 8))
    Z = np.log10(C + 1e-6) if log_scale else C
    contour_fill = ax.contourf(X_km, Y_km, Z, levels=30)
    cbar = plt.colorbar(contour_fill, ax=ax)
    cbar.set_label('log10(concentration + 1e-6)' if log_scale else 'concentration')
    contour_levels = [cfg.threshold_low, cfg.threshold_medium, cfg.threshold_high]
    cs = ax.contour(
        X_km, Y_km, C,
        levels=contour_levels,
        colors='black',
        linewidths=1.5,
        linestyles=['dashed', 'solid', 'solid']
    )
    ax.clabel(cs, inline=True, fontsize=9, fmt={
        cfg.threshold_low: 'Low',
        cfg.threshold_medium: 'Medium',
        cfg.threshold_high: 'High',
    })
    ax.scatter([cfg.source_x_km], [cfg.source_y_km], marker='*', s=240)
    ax.text(cfg.source_x_km + 0.4, cfg.source_y_km + 0.4, 'Source', fontsize=10)
    theta = wind_to_math_angle_rad(cfg.wind_from_deg)
    arrow_len = 5.0
    ax.arrow(cfg.source_x_km, cfg.source_y_km, arrow_len*np.cos(theta), arrow_len*np.sin(theta), width=0.08, length_includes_head=True)
    ax.text(cfg.source_x_km + arrow_len*np.cos(theta) + 0.2, cfg.source_y_km + arrow_len*np.sin(theta) + 0.2, 'Plume direction', fontsize=10)
    if neighborhoods is not None:
        for name, (xk, yk) in neighborhoods.items():
            ax.scatter([xk], [yk], s=60)
            ax.text(xk + 0.2, yk + 0.2, name, fontsize=9)
    ax.set_title(f'Generic Toxic Plume Simulator {title_suffix}\nWind from {cfg.wind_from_deg:.0f}°, speed={cfg.wind_speed_mps:.1f} m/s, stability={cfg.stability_class}, emission={cfg.emission_rate:.0f}')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_xlim(cfg.x_min_km, cfg.x_max_km)
    ax.set_ylim(cfg.y_min_km, cfg.y_max_km)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.show()

def sample_neighborhoods(cfg: PlumeConfig, neighborhoods):
    rows = []
    for name, (xk, yk) in neighborhoods.items():
        X = np.array([[xk]])
        Y = np.array([[yk]])
        x_m, y_m = rotate_to_wind_frame(X, Y, cfg.source_x_km, cfg.source_y_km, cfg.wind_from_deg)
        C = gaussian_plume_ground_level(
            x_m=x_m,
            y_m=y_m,
            Q=cfg.emission_rate,
            U=max(cfg.wind_speed_mps, 0.1),
            H=cfg.release_height_m,
            stability_class=cfg.stability_class,
            min_downwind_m=cfg.min_downwind_m,
        )[0, 0]
        rows.append({
            'Neighborhood': name,
            'X_km': xk,
            'Y_km': yk,
            'Concentration': C,
            'Band': concentration_band(C, cfg),
        })
    return pd.DataFrame(rows).sort_values('Concentration', ascending=False).reset_index(drop=True)

print('SUCCEEDED: Plotting functions created')


In [ ]:
plot_plume(cfg, neighborhoods=FICTIONAL_NEIGHBORHOODS, title_suffix='(Base Scenario)')
base_df = sample_neighborhoods(cfg, FICTIONAL_NEIGHBORHOODS)
base_df


In [ ]:
SCENARIOS = [
    {'Scenario': 'Cool-season westerly', 'wind_from_deg': 270.0, 'wind_speed_mps': 5.0, 'stability_class': 'D', 'emission_rate': 1200.0},
    {'Scenario': 'Warm-season easterly', 'wind_from_deg': 90.0, 'wind_speed_mps': 5.0, 'stability_class': 'D', 'emission_rate': 1200.0},
    {'Scenario': 'Stable night', 'wind_from_deg': 270.0, 'wind_speed_mps': 3.0, 'stability_class': 'F', 'emission_rate': 1200.0},
    {'Scenario': 'Unstable day', 'wind_from_deg': 270.0, 'wind_speed_mps': 7.0, 'stability_class': 'B', 'emission_rate': 1200.0},
]

summary_tables = []
for s in SCENARIOS:
    local_cfg = PlumeConfig(
        x_min_km=cfg.x_min_km, x_max_km=cfg.x_max_km,
        y_min_km=cfg.y_min_km, y_max_km=cfg.y_max_km,
        resolution=cfg.resolution,
        source_x_km=cfg.source_x_km, source_y_km=cfg.source_y_km,
        wind_from_deg=s['wind_from_deg'], wind_speed_mps=s['wind_speed_mps'],
        emission_rate=s['emission_rate'], release_height_m=cfg.release_height_m,
        stability_class=s['stability_class'], min_downwind_m=cfg.min_downwind_m,
        threshold_low=cfg.threshold_low, threshold_medium=cfg.threshold_medium, threshold_high=cfg.threshold_high,
    )
    plot_plume(local_cfg, neighborhoods=FICTIONAL_NEIGHBORHOODS, title_suffix=f"({s['Scenario']})")
    tmp = sample_neighborhoods(local_cfg, FICTIONAL_NEIGHBORHOODS)
    tmp.insert(0, 'Scenario', s['Scenario'])
    summary_tables.append(tmp)

scenario_df = pd.concat(summary_tables, ignore_index=True)
scenario_df.head(20)


In [ ]:
pivot = scenario_df.pivot_table(index='Neighborhood', columns='Scenario', values='Concentration', aggfunc='max')
pivot


In [ ]:
band_pivot = scenario_df.pivot_table(index='Neighborhood', columns='Scenario', values='Band', aggfunc='first')
band_pivot


In [ ]:
def gaussian_puff_ground_level(x_m, y_m, t_s, Q_total, U, H, stability_class='D', t_release_s=0.0):
    if t_s <= t_release_s:
        return np.zeros_like(x_m, dtype=float)
    dt = t_s - t_release_s
    x_center = U * dt
    x_for_sigma = max(x_center, 1.0)
    sy, sz = sigma_yz(np.array([x_for_sigma]), stability_class)
    sy = sy[0]
    sz = sz[0]
    sx = max(sy, 1.0)
    prefactor = Q_total / (((2*np.pi)**1.5) * sx * sy * sz)
    along = np.exp(-((x_m - x_center) ** 2) / (2.0 * sx ** 2))
    cross = np.exp(-(y_m ** 2) / (2.0 * sy ** 2))
    vert = np.exp(-(H ** 2) / (2.0 * sz ** 2))
    return prefactor * along * cross * vert

def compute_puff_concentration(cfg: PlumeConfig, t_s: float, puff_mass=None):
    if puff_mass is None:
        puff_mass = cfg.emission_rate * 60.0
    X_km, Y_km = make_grid(cfg)
    x_m, y_m = rotate_to_wind_frame(X_km, Y_km, cfg.source_x_km, cfg.source_y_km, cfg.wind_from_deg)
    C = gaussian_puff_ground_level(x_m=x_m, y_m=y_m, t_s=t_s, Q_total=puff_mass, U=max(cfg.wind_speed_mps, 0.1), H=cfg.release_height_m, stability_class=cfg.stability_class)
    return X_km, Y_km, C

def plot_puff(cfg: PlumeConfig, t_s: float, neighborhoods=None):
    X_km, Y_km, C = compute_puff_concentration(cfg, t_s=t_s)
    fig, ax = plt.subplots(figsize=(11, 8))
    Z = np.log10(C + 1e-6)
    im = ax.contourf(X_km, Y_km, Z, levels=30)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('log10(concentration + 1e-6)')
    ax.scatter([cfg.source_x_km], [cfg.source_y_km], marker='*', s=240)
    ax.text(cfg.source_x_km + 0.3, cfg.source_y_km + 0.3, 'Source', fontsize=10)
    if neighborhoods is not None:
        for name, (xk, yk) in neighborhoods.items():
            ax.scatter([xk], [yk], s=60)
            ax.text(xk + 0.2, yk + 0.2, name, fontsize=9)
    ax.set_title(f'Generic Puff Release (Educational) at t={t_s/60:.1f} min\nWind from {cfg.wind_from_deg:.0f}°, speed={cfg.wind_speed_mps:.1f} m/s, stability={cfg.stability_class}')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_xlim(cfg.x_min_km, cfg.x_max_km)
    ax.set_ylim(cfg.y_min_km, cfg.y_max_km)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    plt.show()

plot_puff(cfg, t_s=15*60, neighborhoods=FICTIONAL_NEIGHBORHOODS)


In [ ]:
def animate_puff(cfg: PlumeConfig, neighborhoods=None, t_end_s=60*60, frames=40):
    X_km, Y_km = make_grid(cfg)
    times = np.linspace(60, t_end_s, frames)
    fig, ax = plt.subplots(figsize=(11, 8))
    C0 = compute_puff_concentration(cfg, t_s=times[0])[2]
    Z0 = np.log10(C0 + 1e-6)
    contour_holder = [ax.contourf(X_km, Y_km, Z0, levels=30)]
    plt.colorbar(contour_holder[0], ax=ax, label='log10(concentration + 1e-6)')
    ax.scatter([cfg.source_x_km], [cfg.source_y_km], marker='*', s=240)
    ax.text(cfg.source_x_km + 0.3, cfg.source_y_km + 0.3, 'Source', fontsize=10)
    if neighborhoods is not None:
        for name, (xk, yk) in neighborhoods.items():
            ax.scatter([xk], [yk], s=60)
            ax.text(xk + 0.2, yk + 0.2, name, fontsize=9)
    ax.set_xlim(cfg.x_min_km, cfg.x_max_km)
    ax.set_ylim(cfg.y_min_km, cfg.y_max_km)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    def update(frame_idx):
        for coll in contour_holder[0].collections:
            coll.remove()
        t_s = times[frame_idx]
        C = compute_puff_concentration(cfg, t_s=t_s)[2]
        Z = np.log10(C + 1e-6)
        contour_holder[0] = ax.contourf(X_km, Y_km, Z, levels=30)
        ax.set_title(f'Generic Puff Animation (Educational) | t={t_s/60:.1f} min\nWind from {cfg.wind_from_deg:.0f}°, speed={cfg.wind_speed_mps:.1f} m/s, stability={cfg.stability_class}')
        return contour_holder[0].collections
    anim = FuncAnimation(fig, update, frames=len(times), interval=200, blit=False)
    plt.close(fig)
    return anim

# Usage:
# anim = animate_puff(cfg, neighborhoods=FICTIONAL_NEIGHBORHOODS)
# anim


In [ ]:
if HAS_WIDGETS:
    def interactive_plume(wind_from_deg=270.0, wind_speed_mps=5.0, stability_class='D', emission_rate=1200.0, release_height_m=10.0):
        local_cfg = PlumeConfig(
            x_min_km=cfg.x_min_km, x_max_km=cfg.x_max_km,
            y_min_km=cfg.y_min_km, y_max_km=cfg.y_max_km,
            resolution=cfg.resolution,
            source_x_km=cfg.source_x_km, source_y_km=cfg.source_y_km,
            wind_from_deg=wind_from_deg, wind_speed_mps=wind_speed_mps,
            emission_rate=emission_rate, release_height_m=release_height_m,
            stability_class=stability_class,
            min_downwind_m=cfg.min_downwind_m,
            threshold_low=cfg.threshold_low, threshold_medium=cfg.threshold_medium, threshold_high=cfg.threshold_high,
        )
        plot_plume(local_cfg, neighborhoods=FICTIONAL_NEIGHBORHOODS, title_suffix='(Interactive)')
        try:
            from IPython.display import display
            display(sample_neighborhoods(local_cfg, FICTIONAL_NEIGHBORHOODS))
        except Exception:
            print(sample_neighborhoods(local_cfg, FICTIONAL_NEIGHBORHOODS))

    interact(
        interactive_plume,
        wind_from_deg=widgets.FloatSlider(min=0, max=360, step=5, value=270, description='Wind from'),
        wind_speed_mps=widgets.FloatSlider(min=0.5, max=20.0, step=0.5, value=5.0, description='Wind m/s'),
        stability_class=widgets.Dropdown(options=['A', 'B', 'C', 'D', 'E', 'F'], value='D', description='Stability'),
        emission_rate=widgets.FloatLogSlider(base=10, min=1, max=5, step=0.1, value=1200.0, description='Emission'),
        release_height_m=widgets.FloatSlider(min=0, max=100, step=1, value=10, description='Height m'),
    )
else:
    print('FAILED: ipywidgets not installed. Static notebook cells still work.')


In [ ]:
NOTES = '''
Model caveats:
1. This is a simplified educational Gaussian plume / puff model.
2. It assumes flat terrain and steady wind within each scenario.
3. It does not model terrain blocking, buildings, rainout, chemistry, or time-varying meteorology.
4. Thresholds here are arbitrary educational bands, not official health thresholds.
5. Real emergency-response modeling requires validated source terms, real meteorology, and specialist tools.

Recommended safe uses:
- Education
- Emergency-planning concepts
- Comparing generic weather scenarios
- Testing visualization ideas on fictional maps
'''
print(NOTES)
